# Transaction Intelligence — Phase 3: fine-tune the teacher (Colab)

**Step 0 — turn on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator → GPU** (a free T4 is plenty), then **Save**.

This notebook trains DistilBERT on the transaction dataset and prints macro-F1 on **test** and **gold**. The bar to beat (from the TF-IDF baseline): **gold subtype 0.77 / category 0.85**. Run the cells top to bottom.

In [ ]:
# 1) Get the code (idempotent — safe to re-run).
%cd /content
!rm -rf transaction-intelligence
!git clone https://github.com/thejayvaghela/transaction-intelligence.git
%cd transaction-intelligence
# PRIVATE repo? Replace the clone line above with these two:
#   from getpass import getpass; TOKEN = getpass("GitHub token: ")
#   !git clone https://{TOKEN}@github.com/thejayvaghela/transaction-intelligence.git

In [ ]:
# 2) Install dependencies (robust on Colab: explicit deps, then the package without re-resolving).
!pip install -q "transformers>=4.41" datasets accelerate sentencepiece mlflow-skinny
!pip install -q -e . --no-deps

In [ ]:
# 3) Recreate the dataset deterministically (data/ is gitignored; this rebuilds the exact splits).
!python scripts/build_dataset.py

In [ ]:
# 4) (Optional) Mount Google Drive to keep the trained model after the session. Safe to skip.
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/transaction-intelligence'
os.makedirs(DRIVE, exist_ok=True)
print('Drive ready:', DRIVE)

In [ ]:
# 5) Train the teacher (DistilBERT, ~4 epochs). A few minutes on a T4.
#    Prints subtype/category macro-F1 on test + gold, and logs the run to MLflow.
!python scripts/train_transformer.py

In [ ]:
# 6) Save the trained model to Drive (so you can download it / evaluate locally).
!cp -r models/teacher "$DRIVE/teacher"
print('model saved to Drive:', DRIVE + '/teacher')

## After training
1. Read the printed **`teacher/gold`** line — compare its subtype/category macro-F1 to the baseline (0.77 / 0.85).
2. The model is in `models/teacher` and on Drive (`MyDrive/transaction-intelligence/teacher`).
3. Paste the printed numbers back to Claude, and/or download the `teacher` folder to evaluate locally with the same harness.

**To try the stronger DeBERTa-v3 teacher:** edit `configs/finetune.yaml` → `model_id: microsoft/deberta-v3-small`, then re-run cell 5.